# Model 2 - CNN Parallel
Notebook train và test CNN Parallel.

In [1]:
# 1. Import và cấu hình
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import classification_report

from metrics import calculate_metrics

DATA_DIR = Path('../../../data/VN_trash_classification_preprocessing')
CHECKPOINT_DIR = Path('../checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS = 30
PATIENCE = 7
BATCH_SIZE = 32
LR = 1e-4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
# 2. Xử lý ảnh và load dữ liệu
normalize = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    normalize,
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize,
])

train_ds = ImageFolder(DATA_DIR / 'train', transform=train_transform)
val_ds = ImageFolder(DATA_DIR / 'val', transform=test_transform)
test_ds = ImageFolder(DATA_DIR / 'test', transform=test_transform)

# Nạp ảnh bằng nhiều tiến trình để GPU không phải chờ CPU
loader_options = {'num_workers': 8, 'pin_memory': True, 'persistent_workers': True}
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, **loader_options)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, **loader_options)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, **loader_options)

class_names = train_ds.classes
print('Device:', device, '| Classes:', class_names)
print('Train:', len(train_ds), '| Validation:', len(val_ds), '| Test:', len(test_ds))

Device: cuda | Classes: ['Alu', 'Carton', 'Foam_box', 'Milk_box', 'Other', 'PET', 'Paper', 'Paper_cup', 'Plastic_cup']
Train: 9542 | Validation: 1685 | Test: 864


In [3]:
# 3. Tạo CNN Parallel
class ResidualConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, stride=2, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.main(x) + self.shortcut(x))


class CNNParallel(nn.Module):
    def __init__(self, number_of_classes):
        super().__init__()

        # 3 nhánh song song, mỗi nhánh ra 32 kênh
        self.branch1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.branch2 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.branch3 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=2, dilation=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )

        # Sau khi nối: 32 + 32 + 32 = 96 kênh
        self.features = nn.Sequential(
            ResidualConvBlock(96, 64),
            ResidualConvBlock(64, 128),
            ResidualConvBlock(128, 192),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(192, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, number_of_classes),
        )

    def forward(self, images):
        x = torch.cat([self.branch1(images), self.branch2(images), self.branch3(images)], dim=1)
        return self.classifier(self.features(x))


model = CNNParallel(len(class_names)).to(device)
print(model)

CNNParallel(
  (branch1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (branch2): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): ReLU(inplace=True)
  )
  (branch3): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(2, 2), dilation=(2, 2))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (features): Sequential(
    (0): ResidualConvBlock(
      (main): Sequen

In [4]:
# 4. Loss, optimizer và scheduler
counts = Counter(train_ds.targets)
weights = torch.tensor(
    [len(train_ds) / (len(class_names) * counts[i]) for i in range(len(class_names))],
    dtype=torch.float32,
    device=device,
)

loss_fn = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.01)

In [5]:
# 5. Train và validation
def run_epoch(loader, training=False):
    model.train(training)
    total_loss = 0.0
    correct = 0
    seen = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        seen += labels.size(0)

    return total_loss / seen, correct / seen


best_acc = -1.0
best_state = None
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, True)
    val_loss, val_acc = run_epoch(val_loader)
    scheduler.step()

    if val_acc > best_acc:
        best_acc = val_acc
        best_epoch = epoch + 1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f'Epoch {epoch + 1}/{EPOCHS} | '
        f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2%} | '
        f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2%}'
    )

    # Dừng khi validation accuracy không tăng trong PATIENCE epoch liên tiếp.
    if epochs_without_improvement >= PATIENCE:
        print(f'Early stopping tại epoch {epoch + 1}')
        break

model.load_state_dict(best_state)
print('Best epoch:', best_epoch, '| Best validation accuracy:', f'{best_acc:.2%}')

Epoch 1/30 | Train Loss: 1.9236 | Train Acc: 31.03% | Val Loss: 1.8175 | Val Acc: 37.86%


Epoch 2/30 | Train Loss: 1.7684 | Train Acc: 37.83% | Val Loss: 1.7289 | Val Acc: 40.47%


Epoch 3/30 | Train Loss: 1.7014 | Train Acc: 40.56% | Val Loss: 1.6493 | Val Acc: 42.91%


Epoch 4/30 | Train Loss: 1.6172 | Train Acc: 43.93% | Val Loss: 1.5745 | Val Acc: 46.05%


Epoch 5/30 | Train Loss: 1.5699 | Train Acc: 46.79% | Val Loss: 1.5510 | Val Acc: 47.42%


...

Epoch 26/30 | Train Loss: 0.8960 | Train Acc: 71.31% | Val Loss: 0.9378 | Val Acc: 71.45%


Epoch 27/30 | Train Loss: 0.8977 | Train Acc: 70.78% | Val Loss: 0.9436 | Val Acc: 69.14%


Epoch 28/30 | Train Loss: 0.8927 | Train Acc: 70.81% | Val Loss: 0.9330 | Val Acc: 70.92%


Epoch 29/30 | Train Loss: 0.8916 | Train Acc: 71.10% | Val Loss: 0.9232 | Val Acc: 71.04%


Epoch 30/30 | Train Loss: 0.8842 | Train Acc: 71.31% | Val Loss: 0.9428 | Val Acc: 70.45%
Best epoch: 26 | Best validation accuracy: 71.45%


In [6]:
# 6. Test và đánh giá
model.eval()
true_labels = []
predicted_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predictions.cpu().tolist())

metrics = calculate_metrics(true_labels, predicted_labels)

print()
print('Test Result')
print(f'Test Accuracy: {metrics["accuracy"]:.2%}')
print(f'Macro Precision: {metrics["macro_precision"]:.4f}')
print(f'Macro Recall: {metrics["macro_recall"]:.4f}')
print(f'Macro F1: {metrics["macro_f1"]:.4f}')
print(f'Weighted F1: {metrics["weighted_f1"]:.4f}')
print()
print(classification_report(true_labels, predicted_labels, target_names=class_names, digits=4, zero_division=0))


Test Result
Test Accuracy: 64.58%
Macro Precision: 0.6383
Macro Recall: 0.6428
Macro F1: 0.6361
Weighted F1: 0.6459

              precision    recall  f1-score   support

         Alu     0.7500    0.6667    0.7059        99
      Carton     0.8469    0.7685    0.8058       108
    Foam_box     0.4750    0.4368    0.4551        87
    Milk_box     0.8190    0.8515    0.8350       101
       Other     0.4459    0.6600    0.5323        50
         PET     0.6125    0.4804    0.5385       102
       Paper     0.5938    0.7170    0.6496       106
   Paper_cup     0.6916    0.7048    0.6981       105
 Plastic_cup     0.5096    0.5000    0.5048       106

    accuracy                         0.6458       864
   macro avg     0.6383    0.6428    0.6361       864
weighted avg     0.6529    0.6458    0.6459       864



In [7]:
# 7. Lưu model
torch.save(model.state_dict(), CHECKPOINT_DIR / 'cnn_parallel.pt')